# PhysDual-GNN — Bootstrap do Ambiente no Google Colab (Fase 0)

Este notebook é **idempotente** (pode ser reexecutado a qualquer momento após quedas ou reinícios de sessão sem efeitos colaterais). Ele prepara o ambiente no Google Colab (GPU T4 ou L4) e persiste código, cache e resultados no Google Drive.

### Célula 1 — Verificar GPU (`nvidia-smi`)
Verifica se há uma GPU NVIDIA alocada para esta sessão. **Se nenhuma GPU aparecer**, vá no menu superior em **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4 ou L4)**, salve e execute novamente.

In [ ]:
import subprocess

res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print("[AVISO] GPU não detectada! Vá em: Ambiente de execução -> Alterar tipo -> GPU (T4 ou L4).")

### Célula 2 — Montar o Google Drive e definir `PHYSDUAL_ROOT`
Monta o Google Drive em `/content/drive`, define a variável de ambiente `PHYSDUAL_ROOT` (`/content/drive/MyDrive/physdual-gnn`) e garante a criação das pastas persistentes (`data/`, `cache/esm_embeddings/`, `outputs/checkpoints/`, `outputs/results/`).

In [ ]:
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    default_root = "/content/drive/MyDrive/physdual-gnn"
else:
    default_root = str(Path.cwd().resolve())

ROOT = Path(os.environ.get("PHYSDUAL_ROOT", default_root))
os.environ["PHYSDUAL_ROOT"] = str(ROOT)

for sub in ["data/raw", "data/processed", "cache/esm_embeddings", "outputs/checkpoints", "outputs/results"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

print(f"PHYSDUAL_ROOT configurado em: {ROOT}")

### Célula 3 — Clonar ou atualizar o repositório em `{PHYSDUAL_ROOT}/repo`
Se `{PHYSDUAL_ROOT}/repo` ainda não existir, clona o repositório Git; caso já exista, executa `git pull --ff-only` de forma idempotente. Para repositórios privados, utiliza o painel **Secrets** (`google.colab.userdata.get('GITHUB_TOKEN')`) e `GIT_ASKPASS` em memória — **nunca** escreva tokens em texto plano.

In [ ]:
import os
import stat
import subprocess
import sys
import tempfile
from pathlib import Path

ROOT = Path(os.environ["PHYSDUAL_ROOT"])
REPO_DIR = ROOT / "repo"

# Defina a URL do seu repositório (ex.: "https://github.com/<usuario>/physdual-gnn.git")
# ou use o Secret GITHUB_REPO_URL no painel lateral do Colab.
github_token = None
repo_url = os.environ.get("GITHUB_REPO_URL", "")
if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
    if not repo_url:
        try:
            repo_url = userdata.get("GITHUB_REPO_URL") or ""
        except Exception:
            repo_url = ""

def _run_git_secure(args: list[str], cwd: Path | None = None) -> None:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if github_token:
        with tempfile.TemporaryDirectory() as tmp:
            askpass = Path(tmp) / "askpass.sh"
            askpass.write_text('#!/bin/sh\necho "$ COLAB_GH_TOKEN"\n'.replace("$ COLAB_GH_TOKEN", "$COLAB_GH_TOKEN"), encoding="utf-8")
            askpass.chmod(stat.S_IRWXU)
            env["GIT_ASKPASS"] = str(askpass)
            env["COLAB_GH_TOKEN"] = github_token
            subprocess.run(["git", *args], cwd=cwd, env=env, check=True)
    else:
        subprocess.run(["git", *args], cwd=cwd, env=env, check=True)

if (REPO_DIR / ".git").exists():
    print(f"Repositório já existe em {REPO_DIR}. Verificando atualizações...")
    rem = subprocess.run(["git", "remote"], cwd=REPO_DIR, capture_output=True, text=True)
    if rem.stdout.strip():
        _run_git_secure(["pull", "--ff-only"], cwd=REPO_DIR)
    else:
        print("Nenhum remote configurado localmente; mantendo árvore atual em", REPO_DIR)
elif (ROOT / "src" / "physdual").exists():
    REPO_DIR = ROOT
    print(f"Código-fonte detectado diretamente em {REPO_DIR}.")
elif repo_url:
    print(f"Clonando repositório em {REPO_DIR}...")
    _run_git_secure(["clone", repo_url, str(REPO_DIR)])
else:
    raise RuntimeError(
        f"Repositório não encontrado em {REPO_DIR}. "
        "Faça upload da pasta repo para o Drive ou defina GITHUB_REPO_URL nos Secrets do Colab."
    )

os.chdir(REPO_DIR)
print("Diretório ativo do repositório:", Path.cwd())

### Célula 4 — Instalar dependências do Colab (`requirements-colab.txt` + `torch_geometric`)
Instala apenas os pacotes que o Google Colab **não** traz por padrão (`rdkit`, `fair-esm`, `biopython`, `torch_geometric`, `lifelines`, etc.), **sem reinstalar nem alterar o `torch` pré-instalado**.

In [ ]:
import subprocess
import sys
import torch

torch_before = torch.__version__
print(f"PyTorch pré-instalado antes do pip: {torch_before} (CUDA: {torch.version.cuda})")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)

print("Dependências de requirements-colab.txt instaladas com sucesso (torch preservado).")

### Célula 5 — Adicionar `{ROOT}/repo/src` ao `sys.path` e instalar o pacote em modo editável
Garante que o pacote `physdual` seja importável imediatamente em qualquer célula ou script, usando `--no-deps` apenas no `pip install -e .` para que o `setuptools` não tente resolver dependências novamente.

In [ ]:
import subprocess
import sys
from pathlib import Path

src_path = str((Path.cwd() / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)

import physdual
print(f"Pacote physdual v{physdual.__version__} carregado de: {physdual.__file__}")

### Célula 6 — Rodar `diagnose_env.py` e gravar `docs/colab_env_snapshot.txt`
Imprime o diagnóstico completo do ambiente (SO, Python, PyTorch, CUDA, GPU/VRAM, RAM, espaço em disco local e no Drive) e salva o snapshot reprodutível com `pip freeze` em `docs/colab_env_snapshot.txt`.

In [ ]:
!python scripts/diagnose_env.py --snapshot docs/colab_env_snapshot.txt

### Célula 7 — Executar o Smoke Test (`scripts/smoke_test.py`)
Valida os 8 critérios de pronto da Fase 0: imports, detecção de GPU, conversão SMILES $\rightarrow$ grafo PyG via RDKit, forward+backward de `GINConv`, inferência e cache do `esm2_t6_8M_UR50D`, determinismo de `set_seed`, `PyTDC` (opcional) e escrita persistente no Google Drive.

In [ ]:
!python scripts/smoke_test.py

### Célula 8 — Aviso Importante sobre Reinício do Runtime

> **Atenção:** Se o `pip` tiver exibido qualquer aviso indicando que o `numpy`, `scipy` ou outra biblioteca C-extension já carregada em memória foi substituída durante a Célula 4, clique em **Ambiente de execução → Reiniciar sessão** (*Runtime → Restart session*) e **reexecute este notebook a partir da Célula 2**. Como os pacotes já estarão instalados na VM, a segunda execução levará apenas alguns segundos.